In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import argparse
import logging
import warnings
import matplotlib

In [2]:
# Define your data directory path
data_dir = "Data"

# 1. Load the Official Sector Mapping CSV
df_sectors = pd.read_csv(
    os.path.join(data_dir, "NSE_data_stock_market_sectors_2025 (1).csv")
)

# 2. Load each yearly cleaned Excel file individually
df_2019 = pd.read_excel(os.path.join(data_dir, "NSE_Stocks_2019_Cleaned_v2.xlsx"))
df_2020 = pd.read_excel(os.path.join(data_dir, "NSE_Stocks_2020_Cleaned_v2.xlsx"))
df_2021 = pd.read_excel(os.path.join(data_dir, "NSE_Stocks_2021_Cleaned_v2.xlsx"))
df_2022 = pd.read_excel(os.path.join(data_dir, "NSE_Stocks_2022_Cleaned_v2.xlsx"))
df_2023 = pd.read_excel(os.path.join(data_dir, "NSE_Stocks_2023_Cleaned_v2.xlsx"))
df_2024 = pd.read_excel(os.path.join(data_dir, "NSE_Stocks_2024_Cleaned_v2.xlsx"))
df_2025 = pd.read_excel(os.path.join(data_dir, "NSE_Stocks_2025_Cleaned_v2.xlsx"))

# Quick verification that all individual datasets loaded correctly
print("Sector Mapping Rows:", len(df_sectors))
print("2019 Rows:", len(df_2019))
print("2020 Rows:", len(df_2020))
print("2021 Rows:", len(df_2021))
print("2022 Rows:", len(df_2022))
print("2023 Rows:", len(df_2023))
print("2024 Rows:", len(df_2024))
print("2025 Rows:", len(df_2025))

Sector Mapping Rows: 75
2019 Rows: 18147
2020 Rows: 17892
2021 Rows: 17746
2022 Rows: 16806
2023 Rows: 17274
2024 Rows: 18119
2025 Rows: 15114


In [3]:
# 1. Put all yearly dataframes into a list and combine them into one master dataframe
yearly_dfs = [
    df_2019,
    df_2020,
    df_2021,
    df_2022,
    df_2023,
    df_2024,
    df_2025,
]
df_master = pd.concat(yearly_dfs, ignore_index=True)

# 2. Merge df_master with the sector mapping file using the matching ticker columns ('Code' and 'Stock_code')
df_master = pd.merge(
    df_master,
    df_sectors[['Stock_code', 'Sector']],
    left_on='Code',
    right_on='Stock_code',
    how='left',
)

# 3. Check the results
print('Master Dataset Shape:', df_master.shape)
print('Columns after merge:', df_master.columns.tolist())
print(
    'Missing Sectors Count:',
    df_master['Sector'].isna().sum(),
)  # Should be 0 or very low if all tickers match

Master Dataset Shape: (121098, 17)
Columns after merge: ['Date', 'Code', 'Name', '12m Low', '12m High', 'Day Low', 'Day High', 'Day Price', 'Previous', 'Change', 'Change%', 'Volume', 'Price Imputed', 'Volume Imputed', 'Asset_Type', 'Stock_code', 'Sector']
Missing Sectors Count: 5550


In [4]:
df_master.head(10)

,Date,Code,Name,12m Low,12m High,Day Low,Day High,Day Price,Previous,Change,Change%,Volume,Price Imputed,Volume Imputed,Asset_Type,Stock_code,Sector
0,2019-01-02,ABSA,ABSA Bank Kenya Plc,8.96,14.20,10.80,11.00,10.85,10.95,-0.10,-0.009132,120200,0,0,Stock,ABSA,Banking
1,2019-01-02,ARM,ARM Cement Plc,5.55,5.55,5.55,5.55,5.55,5.55,0.00,0.000000,0,0,1,Stock,ARM,Construction and Allied
2,2019-01-02,BAMB,Bamburi Cement Ltd,24.90,115.00,132.50,132.50,132.50,132.50,0.00,0.000000,0,0,1,Stock,BAMB,Construction and Allied
3,2019-01-02,BAT,British American Tobacco Kenya Plc,305.00,545.00,725.00,725.00,725.00,725.00,0.00,0.000000,0,0,1,Stock,BAT,Manufacturing and Allied
4,2019-01-02,BKG,BK Group Plc,16.40,28.00,33.00,33.00,33.00,33.00,0.00,0.000000,0,0,1,Stock,BKG,Banking
5,2019-01-02,BOC,BOC Kenya Plc,53.00,72.00,75.00,75.00,75.00,75.00,0.00,0.000000,0,0,1,Stock,BOC,Manufacturing and Allied
6,2019-01-02,BRIT,Britam Holdings Plc,5.70,10.65,10.00,10.50,10.05,10.00,0.05,0.005000,101900,0,0,Stock,BRIT,Insurance
7,2019-01-02,CABL,East African Cables Plc,1.44,3.98,2.60,2.75,2.65,2.72,-0.07,-0.025735,1600,0,0,Stock,CABL,Construction and Allied
8,2019-01-02,CARB,Carbacid Investments Plc,7.10,9.40,10.00,10.00,10.00,10.00,0.00,0.000000,2600,0,0,Stock,CARB,Manufacturing and Allied
9,2019-01-02,CGEN,Car and General (K) Ltd,20.00,26.00,23.00,23.50,23.45,21.50,1.95,0.090698,4300,0,0,Stock,CGEN,Automobiles and Accessories


### Macro-Shock & Crisis Window Analysis

**COVID-19 Pandemic Shock (Q1–Q2 2020):**
- Highlighted severe capital flight as foreign investors liquidated frontier-market assets, causing broad-based panic selling and steep double-digit drawdowns across liquidity-heavy counters before state monetary interventions took effect.

- Defensive Sector Resilience: The agricultural sector and select defensive consumer goods counters acted as safe havens, displaying lower downward velocity during pandemic lockdowns due to steady domestic demand for essential commodities.

**2022 General Election Volatility:**
- Demonstrated typical pre-election jitters where political uncertainty triggered compressed trading volumes and cautious capital deployment, though market recovery was relatively swift post-election compared to macro-driven shocks.

**2023 Macroeconomic Tightening:**
- Exposed the severe vulnerability of highly leveraged stocks to surging inflation, aggressive central bank rate hikes, and severe foreign exchange pressures stemming from the Finance Act reforms.

**Liquidity Divergence:**
- Confirmed that macroeconomic shocks disproportionately punish illiquid counters (high zero-volume percentage days), whereas top-tier blue-chip stocks absorb shocks much faster due to deeper institutional liquidity pools.

### 1. COVID-19 Pandemic Crash (2020-03-01 to 2020-07-31)

In [5]:
# Define COVID-19 Window
start_date, end_date = '2020-03-01', '2020-07-31'
df_window = df_master[
    (df_master['Date'] >= start_date) & (df_master['Date'] <= end_date)
].copy()

# Sort and get first/last prices per company within the window
df_window = df_window.sort_values(by=['Name', 'Date'])
start_p = df_window.groupby('Name').first().reset_index()
end_p = df_window.groupby('Name').last().reset_index()

covid_detailed = pd.merge(
    start_p[['Name', 'Sector', 'Date', 'Day Price']],
    end_p[['Name', 'Date', 'Day Price']],
    on='Name',
    suffixes=('_start', '_end'),
)

# Calculate comprehensive metrics
covid_detailed['Price_Change_KES'] = (
    covid_detailed['Day Price_end'] - covid_detailed['Day Price_start']
)
covid_detailed['Window_Return_Pct'] = (
    covid_detailed['Price_Change_KES'] / covid_detailed['Day Price_start']
) * 100

# Sort by worst hit (lowest return)
covid_detailed = covid_detailed.sort_values(
    by='Window_Return_Pct', ascending=True
)

print('==================================================================')
print('   COVID-19 CRASH: COMPREHENSIVE COMPANY-WISE SHOCK ANALYSIS       ')
print('==================================================================')
print(
    covid_detailed[
        [
            'Name',
            'Sector',
            'Day Price_start',
            'Day Price_end',
            'Price_Change_KES',
            'Window_Return_Pct',
        ]
    ].to_string(index=False)
)

   COVID-19 CRASH: COMPREHENSIVE COMPANY-WISE SHOCK ANALYSIS       
                                    Name                      Sector  Day Price_start  Day Price_end  Price_Change_KES  Window_Return_Pct
                  Nation Media Group Plc     Commercial and Services            29.50          10.55            -18.95         -64.237288
                      Bamburi Cement Ltd     Construction and Allied            56.00          25.50            -30.50         -54.464286
                       Express Kenya Plc     Commercial and Services             6.50           3.90             -2.60         -40.000000
            Diamond Trust Bank Kenya Ltd                     Banking           102.00          65.75            -36.25         -35.539216
         Nairobi Securities Exchange Plc         Investment Services            10.50           6.80             -3.70         -35.238095
      British American Tobacco Kenya Plc    Manufacturing and Allied           485.00         328.00    

**Day Price_start:** The stock's baseline price in Kenyan Shillings (KES) at the exact beginning of the crisis window (2020-03-01, just as pandemic panic hit the market).
**Day Price_end:** The stock's ending price in KES at the close of the crisis window (2020-07-31, reflecting the depth of the crash or the speed of the initial rebound).

**Price_Change_KES:** The absolute monetary gain or loss in Kenyan Shillings (Day Price_end −Day Price_start)
- A negative number means the stock lost value in shillings, while a positive number means it gained.

**Window_Return_Pct:** The percentage return (
%) over the entire crisis window (
Price_Change_KES
Day Price_start
×
100
Day Price_start
Price_Change_KES
​
 ×100).

- Negative percentages (e.g., -15.5%) represent a crash/drawdown, indicating how much investor capital was eroded.

- Positive percentages (e.g., +5.2%) indicate that a company bucked the trend, managing to grow or hold its value despite the pandemic panic.

In [6]:
# 1. Get Top 5 Best Performers (Highest positive return or least negative)
top_best_covid = covid_detailed.sort_values(
    by='Window_Return_Pct', ascending=False
).head(10)

# 2. Get Top 5 Worst Performers (Deepest crash / lowest return)
top_worst_covid = covid_detailed.sort_values(
    by='Window_Return_Pct', ascending=True
).head(10)

# Display Results
print('==================================================================')
print('        COVID-19 SHOCK: TOP 10 BEST PERFORMING COMPANIES           ')
print('==================================================================')
print(
    top_best_covid[
        [
            'Name',
            'Sector',
            'Day Price_start',
            'Day Price_end',
            'Window_Return_Pct',
        ]
    ].to_string(index=False)
)

print('\n==================================================================')
print('        COVID-19 SHOCK: TOP 10 WORST PERFORMING COMPANIES          ')
print('==================================================================')
print(
    top_worst_covid[
        [
            'Name',
            'Sector',
            'Day Price_start',
            'Day Price_end',
            'Window_Return_Pct',
        ]
    ].to_string(index=False)
)

        COVID-19 SHOCK: TOP 10 BEST PERFORMING COMPANIES           
                    Name                  Sector  Day Price_start  Day Price_end  Window_Return_Pct
       Kenya Airways Plc Commercial and Services             2.15           3.83          78.139535
       Sameer Africa Plc Commercial and Services             2.53           3.48          37.549407
        ABSA NewGold ETF   Exchange Traded Funds          1555.00        1980.00          27.331190
  Uchumi Supermarket Plc Commercial and Services             0.28           0.35          25.000000
              Kakuzi Plc            Agricultural           319.00         395.75          24.059561
 East African Cables Plc Construction and Allied             1.95           2.36          21.025641
      Standard Group Plc Commercial and Services            19.50          23.00          17.948718
Williamson Tea Kenya Plc            Agricultural           122.75         144.25          17.515275
 Kapchorua Tea Kenya Plc        

#### 2. The 2022 General Elections & Rate Hikes (2022-06-01 to 2022-10-31).

**During the 2022 general election window, buying and selling on the Nairobi Securities Exchange (NSE) slowed down significantly as major local and foreign investors held back their money due to political nervousness, rising everyday prices, and global shipping delays.**

* **Compressed Trading Activity:** Instead of normal, lively buying and selling, the stock market became quiet and squeezed. Fewer shares changed hands because both buyers and sellers were hesitant to make big moves, waiting on the sidelines to see what would happen next.
* **Domestic and International Institutional Investors:** These are the "big players" who move millions or billions of shillings—such as local pension funds, insurance companies, and banks (**domestic**), alongside large foreign investment firms and global funds (**international**). When these big players pause their buying or pull money out to safer investments like government bonds, the entire stock market loses momentum.
* **Election Uncertainty:** Ahead of Kenya's August 2022 vote, investors did not know who would win, whether the transition would be peaceful, or if a new government would change taxes and business rules. Because money dislikes guesswork, big investors paused new investments until the political dust settled.
* **Rising Inflation:** Inflation means the general cost of living and doing business goes up—such as higher fuel, electricity, and food prices. When inflation rises, ordinary consumers have less spare cash to spend, and companies face higher operating costs, which squeezes business profits and makes their shares less attractive to buy.
* **Global Supply Chain Pressures:** Due to the lingering aftershocks of COVID-19 and the Russia-Ukraine war in 2022, shipping goods, raw materials, and fuel around the world became slow and expensive. Kenyan businesses that rely on importing goods or machinery faced delays and higher costs, further hurting investor confidence in the stock market.

In [7]:
# 1. Define the 2022 General Elections Window
start_date, end_date = '2022-06-01', '2022-10-31'
df_window = df_master[
    (df_master['Date'] >= start_date) & (df_master['Date'] <= end_date)
].copy()

# 2. Sort chronologically and isolate start/end prices per company
df_window = df_window.sort_values(by=['Name', 'Date'])
start_p = df_window.groupby('Name').first().reset_index()
end_p = df_window.groupby('Name').last().reset_index()

election_detailed = pd.merge(
    start_p[['Name', 'Sector', 'Date', 'Day Price']],
    end_p[['Name', 'Date', 'Day Price']],
    on='Name',
    suffixes=('_start', '_end'),
)

# 3. Calculate monetary change (KES) and percentage return (%)
election_detailed['Price_Change_KES'] = (
    election_detailed['Day Price_end'] - election_detailed['Day Price_start']
)
election_detailed['Window_Return_Pct'] = (
    election_detailed['Price_Change_KES'] / election_detailed['Day Price_start']
) * 100

# 4. Extract Top 10 Best and Top 10 Worst Performers
top_best_election = election_detailed.sort_values(
    by='Window_Return_Pct', ascending=False
).head(10)
top_worst_election = election_detailed.sort_values(
    by='Window_Return_Pct', ascending=True
).head(10)

# 5. Display Comprehensive Results
print('==================================================================')
print('     2022 ELECTIONS: TOP 10 BEST PERFORMING COMPANIES              ')
print('==================================================================')
print(
    top_best_election[
        [
            'Name',
            'Sector',
            'Day Price_start',
            'Day Price_end',
            'Price_Change_KES',
            'Window_Return_Pct',
        ]
    ].to_string(index=False)
)

print('\n==================================================================')
print('     2022 ELECTIONS: TOP 10 WORST PERFORMING COMPANIES             ')
print('==================================================================')
print(
    top_worst_election[
        [
            'Name',
            'Sector',
            'Day Price_start',
            'Day Price_end',
            'Price_Change_KES',
            'Window_Return_Pct',
        ]
    ].to_string(index=False)
)

print('\n==================================================================')
print('     2022 ELECTIONS: FULL COMPANY-WISE BREAKDOWN (ALL STOCKS)     ')
print('==================================================================')
print(
    election_detailed.sort_values(by='Window_Return_Pct', ascending=True)[
        [
            'Name',
            'Sector',
            'Day Price_start',
            'Day Price_end',
            'Price_Change_KES',
            'Window_Return_Pct',
        ]
    ].to_string(index=False)
)

     2022 ELECTIONS: TOP 10 BEST PERFORMING COMPANIES              
                                Name                   Sector  Day Price_start  Day Price_end  Price_Change_KES  Window_Return_Pct
     TPS Eastern Africa (Serena) Ltd  Commercial and Services            11.60          16.00              4.40          37.931034
              Uchumi Supermarket Plc  Commercial and Services             0.20           0.27              0.07          35.000000
            Williamson Tea Kenya Plc             Agricultural           127.25         164.25             37.00          29.076621
Kenya Power and Lighting Company Plc     Energy and Petroleum             1.50           1.86              0.36          24.000000
        Olympia Capital Holdings Ltd               Investment             2.01           2.49              0.48          23.880597
                      NCBA Group Plc                  Banking            25.85          31.50              5.65          21.856867
             Ka

### Deep-Dive Analysis: 2022 General Election Market Dynamics
**While broad window returns capture overall price movement during the 2022 election cycle, evaluating market stress requires looking beyond simple start-to-end prices. Against a macroeconomic backdrop where overall NSE equity turnover fell by 31% across 2022 due to institutional caution, this section dissects market behavior across three specialized dimensions:**

**Pre-Election vs. Post-Election Split ("Relief Rally" Check):** By splitting the June–October 2022 timeline directly at August 9, 2022 (Election Day), we isolate pre-vote defensive selling ("pre-election jitters") from post-vote buying rebounds ("relief rallies"), measuring how rapidly investor confidence returned to specific sectors once political uncertainty cleared.

**Liquidity Freeze & Volume Drop (% Change in Turnover):** Because flat stock prices can mask a complete absence of buyers, comparing Average Daily Turnover (KES) during the election window against a normal pre-election baseline (January–April 2022) quantifies the exact liquidity freeze—distinguishing stocks that maintained active capital flow from those where trading dried up.

**Intraday / Window Volatility Spike:** Comparing the standard deviation of daily returns during the election period against the remainder of 2022 identifies which sectors experienced sharp daily price turbulence around election petitions and Supreme Court rulings versus those that remained structurally steady.

In [8]:
# 1. Split 2022 Election Window at Election Day (August 9, 2022)
election_day = '2022-08-09'

pre_elec = df_master[
    (df_master['Date'] >= '2022-06-01') & (df_master['Date'] < election_day)
].copy()
post_elec = df_master[
    (df_master['Date'] >= election_day) & (df_master['Date'] <= '2022-10-31')
].copy()


# Helper function to compute start/end prices and percentage returns per sub-window
def get_window_stats(df_sub, prefix):
  df_sub = df_sub.sort_values(by=['Name', 'Date'])
  first_p = (
      df_sub.groupby('Name')['Day Price']
      .first()
      .reset_index(name=f'{prefix}_Start_KES')
  )
  last_p = (
      df_sub.groupby('Name')['Day Price']
      .last()
      .reset_index(name=f'{prefix}_End_KES')
  )

  stats = pd.merge(first_p, last_p, on='Name')
  stats[f'{prefix}_Return_Pct'] = (
      (stats[f'{prefix}_End_KES'] - stats[f'{prefix}_Start_KES'])
      / stats[f'{prefix}_Start_KES']
  ) * 100
  return stats


# 2. Calculate Pre-Election and Post-Election metrics
pre_stats = get_window_stats(pre_elec, 'Pre')
post_stats = get_window_stats(post_elec, 'Post')

# 3. Merge with Sector info and calculate Net Election Swing & Overall Window Return
sectors = df_master[['Name', 'Sector']].drop_duplicates()
elec_split = sectors.merge(pre_stats, on='Name').merge(post_stats, on='Name')

# Relief Swing: Post-election return minus Pre-election return (positive = strong recovery bounce)
elec_split['Relief_Swing_Pct'] = (
    elec_split['Post_Return_Pct'] - elec_split['Pre_Return_Pct']
)

# Overall Window Return: From June 1 start price to Oct 31 end price
elec_split['Overall_Return_Pct'] = (
    (elec_split['Post_End_KES'] - elec_split['Pre_Start_KES'])
    / elec_split['Pre_Start_KES']
) * 100

# Select clean display columns
display_cols = [
    'Name',
    'Sector',
    'Pre_Return_Pct',
    'Post_Return_Pct',
    'Relief_Swing_Pct',
    'Overall_Return_Pct',
]

# 4. Print Top 10 Strongest Post-Election Rebounds (Best Performers)
print('======================================================================')
print('   2022 ELECTIONS: TOP 10 STRONGEST POST-ELECTION REBOUNDS (BEST)     ')
print('======================================================================')
print(
    elec_split.sort_values(by='Relief_Swing_Pct', ascending=False)[display_cols]
    .head(10)
    .to_string(index=False)
)

# 5. Print Top 10 Worst Performing Companies (Deepest Drops / Weakest Recovery)
print('\n======================================================================')
print('   2022 ELECTIONS: TOP 10 WORST PERFORMING COMPANIES (DEEPEST DROPS)  ')
print('======================================================================')
print(
    elec_split.sort_values(by='Overall_Return_Pct', ascending=True)[
        display_cols
    ]
    .head(10)
    .to_string(index=False)
)

   2022 ELECTIONS: TOP 10 STRONGEST POST-ELECTION REBOUNDS (BEST)     
                                Name                   Sector  Pre_Return_Pct  Post_Return_Pct  Relief_Swing_Pct  Overall_Return_Pct
Kenya Power and Lighting Company Plc     Energy and Petroleum       -6.666667        24.832215         31.498881           24.000000
       Nairobi Business Ventures Ltd  Commercial and Services      -15.365854        11.815562         27.181416           -5.365854
                          Kakuzi Plc             Agricultural       -8.395522        14.052953         22.448476            4.477612
                      NCBA Group Plc                  Banking       -0.580271        21.621622         22.201892           21.856867
                       BOC Kenya Plc Manufacturing and Allied       -4.109589        15.087719         19.197308           12.328767
            Carbacid Investments Plc Manufacturing and Allied       -0.421941        12.554113         12.976053            9.70464

**Pre_Return_Pct:** Percentage gain or loss from June 1 up to Election Day (August 9), capturing pre-vote jitters.

**Post_Return_Pct:** Percentage gain or loss from Election Day through October 31, capturing the post-vote reaction.

**Relief_Swing_Pct:** The difference between post-election and pre-election performance (Post_Return_Pct - Pre_Return_Pct). A large positive number flags companies that dropped before the election and staged a sharp rebound afterward.

**Overall_Return_Pct:** The net cumulative percentage change across the entire June–October election cycle, used to rank the worst overall casualties.

## 3. The 2023 Macro Tightening & Tax Reforms (2023-05-01 to 2023-11-30).

**Macroeconomic Backdrop (May – November 2023)**
**During this window, the NSE faced intense fiscal and monetary drawbacks—driving the NSE 20 Share Index down toward 1,416 points and pushing the broader NSE All Share Index (NASI) below 90 points:**

**Aggressive Interest Rate Hikes:** The Central Bank of Kenya (CBK) tightened monetary policy to combat sticky inflation, pushing Treasury bill and bond yields past 15%–16%. This drew institutional capital out of equities into risk-free government debt.

**The Finance Act 2023 & Tax Reforms:** New tax measures (including higher excise duties, housing levies, and capital gains adjustments) squeezed disposable consumer income and increased corporate operating costs.

**Severe Shilling Depreciation & Foreign Outflows:** Persistent weakening of the Kenyan Shilling against the US Dollar eroded dollar-denominated returns, accelerating net foreign investor exits from liquid blue-chip counters.

In [9]:
# Ensure Date is datetime and Daily_Turnover exists
df_master['Date'] = pd.to_datetime(df_master['Date'])
if 'Daily_Turnover' not in df_master.columns:
  df_master['Daily_Turnover'] = df_master['Day Price'] * df_master['Volume']
if 'Daily_Return' not in df_master.columns:
  df_master = df_master.sort_values(by=['Name', 'Date'])
  df_master['Daily_Return'] = df_master.groupby('Name')['Day Price'].pct_change()

# 1. Define 2023 Macro Tightening Window & Pre-Crisis Baseline (Jan - Apr 2023)
start_date, end_date = '2023-05-01', '2023-11-30'
tax_effective_date = '2023-07-01'  # Finance Act 2023 implementation milestone

df_baseline = df_master[
    (df_master['Date'] >= '2023-01-01') & (df_master['Date'] < start_date)
].copy()
df_macro = df_master[
    (df_master['Date'] >= start_date) & (df_master['Date'] <= end_date)
].copy()

# Sub-windows: Pre-Tax Implementation (May-Jun) vs. Post-Tax & Rate Hike Peak (Jul-Nov)
df_pre_tax = df_macro[df_macro['Date'] < tax_effective_date].copy()
df_post_tax = df_macro[df_macro['Date'] >= tax_effective_date].copy()


# Helper function for window price stats
def compute_price_metrics(df_sub, prefix):
  df_sub = df_sub.sort_values(by=['Name', 'Date'])
  first_p = (
      df_sub.groupby('Name')['Day Price']
      .first()
      .reset_index(name=f'{prefix}_Start_KES')
  )
  last_p = (
      df_sub.groupby('Name')['Day Price']
      .last()
      .reset_index(name=f'{prefix}_End_KES')
  )
  res = pd.merge(first_p, last_p, on='Name')
  res[f'{prefix}_Change_KES'] = (
      res[f'{prefix}_End_KES'] - res[f'{prefix}_Start_KES']
  )
  res[f'{prefix}_Return_Pct'] = (
      res[f'{prefix}_Change_KES'] / res[f'{prefix}_Start_KES']
  ) * 100
  return res


# 2. Compute Overall Window, Pre-Tax, and Post-Tax Returns
macro_overall = compute_price_metrics(df_macro, 'Macro')
pre_tax_stats = compute_price_metrics(df_pre_tax, 'PreTax')
post_tax_stats = compute_price_metrics(df_post_tax, 'PostTax')

# 3. Compute Liquidity Shift (Avg Daily Turnover vs Jan-Apr Baseline) & Volatility
base_turnover = (
    df_baseline.groupby('Name')['Daily_Turnover']
    .mean()
    .reset_index(name='Baseline_Turnover_KES')
)
macro_turnover = (
    df_macro.groupby('Name')['Daily_Turnover']
    .mean()
    .reset_index(name='Crisis_Turnover_KES')
)

macro_vol = (
    df_macro.groupby('Name')['Daily_Return']
    .std()
    .reset_index(name='Daily_Volatility')
)
macro_vol['Crisis_Ann_Volatility_Pct'] = (
    macro_vol['Daily_Volatility'] * np.sqrt(252) * 100
)

# 4. Merge All Metrics into Master 2023 Macro Table
sectors = df_master[['Name', 'Sector']].drop_duplicates()

macro_detailed = (
    sectors.merge(macro_overall, on='Name')
    .merge(pre_tax_stats[['Name', 'PreTax_Return_Pct']], on='Name', how='left')
    .merge(
        post_tax_stats[['Name', 'PostTax_Return_Pct']], on='Name', how='left'
    )
    .merge(base_turnover, on='Name', how='left')
    .merge(macro_turnover, on='Name', how='left')
    .merge(macro_vol[['Name', 'Crisis_Ann_Volatility_Pct']], on='Name', how='left')
)

# Calculate % change in liquidity turnover
macro_detailed['Turnover_Change_Pct'] = (
    (
        macro_detailed['Crisis_Turnover_KES']
        - macro_detailed['Baseline_Turnover_KES']
    )
    / macro_detailed['Baseline_Turnover_KES']
) * 100

# Select columns for clean display
summary_cols = [
    'Name',
    'Sector',
    'Macro_Start_KES',
    'Macro_End_KES',
    'Macro_Change_KES',
    'Macro_Return_Pct',
    'PreTax_Return_Pct',
    'PostTax_Return_Pct',
    'Turnover_Change_Pct',
    'Crisis_Ann_Volatility_Pct',
]

# 5. Print Top 10 Best Performing Companies (Most Resilient / Positive Gains)
print(
    '=========================================================================================================='
)
print(
    '          2023 MACRO TIGHTENING: TOP 10 BEST PERFORMING COMPANIES (MOST RESILIENT)                         '
)
print(
    '=========================================================================================================='
)
print(
    macro_detailed.sort_values(by='Macro_Return_Pct', ascending=False)[
        summary_cols
    ]
    .head(10)
    .to_string(index=False)
)

# 6. Print Top 10 Worst Performing Companies (Deepest Macro/FX/Rate Casualties)
print(
    '\n=========================================================================================================='
)
print(
    '          2023 MACRO TIGHTENING: TOP 10 WORST PERFORMING COMPANIES (DEEPEST DROPS)                        '
)
print(
    '=========================================================================================================='
)
print(
    macro_detailed.sort_values(by='Macro_Return_Pct', ascending=True)[
        summary_cols
    ]
    .head(10)
    .to_string(index=False)
)

# 7. Print Sector-Level Summary
sector_macro = (
    macro_detailed.groupby('Sector')
    .agg(
        Companies=('Name', 'count'),
        Avg_Return_Pct=('Macro_Return_Pct', 'mean'),
        Median_Return_Pct=('Macro_Return_Pct', 'median'),
        Avg_Turnover_Change_Pct=('Turnover_Change_Pct', 'mean'),
        Avg_Volatility_Pct=('Crisis_Ann_Volatility_Pct', 'mean'),
    )
    .sort_values(by='Avg_Return_Pct', ascending=False)
    .reset_index()
)

print(
    '\n=========================================================================================================='
)
print(
    '          2023 MACRO TIGHTENING: SECTOR-LEVEL RESILIENCE SUMMARY                                          '
)
print(
    '=========================================================================================================='
)
print(sector_macro.to_string(index=False))

          2023 MACRO TIGHTENING: TOP 10 BEST PERFORMING COMPANIES (MOST RESILIENT)                         
                            Name                  Sector  Macro_Start_KES  Macro_End_KES  Macro_Change_KES  Macro_Return_Pct  PreTax_Return_Pct  PostTax_Return_Pct  Turnover_Change_Pct  Crisis_Ann_Volatility_Pct
         Kapchorua Tea Kenya Plc            Agricultural           125.75         239.25            113.50         90.258449          49.701789           24.771838           495.940193                  43.612327
        Eveready East Africa Ltd Commercial and Services             0.78           1.31              0.53         67.948718          84.615385           -2.962963           702.655469                  75.246422
              Bamburi Cement Ltd Construction and Allied            23.95          34.75             10.80         45.093946           5.219207           33.653846           -58.391423                  45.422031
              Kenya Orchards Ltd            

**PreTax_Return_Pct vs. PostTax_Return_Pct:** Separates initial budget anticipation (May–June) from the actual shock once the Finance Act 2023 and CBK rate hikes hit corporate margins and consumer spending (July–November).

**Turnover_Change_Pct:** Shows whether trading activity dried up or spiked in panic compared to the Jan–Apr 2023 baseline.

**Crisis_Ann_Volatility_Pct:** Flags which stocks suffered the most violent daily price swings during currency depreciation and foreign sell-offs.

### Macro-Economic Changes that happened in 2023

1. **The "T-Bill Crowding-Out" Opportunity Cost (Equity vs. Risk-Free Debt)** In late 2023, 91-day and 364-day Kenyan Treasury Bill rates surged past 15%–16%, creating a massive "crowding-out" effect where local institutional investors abandoned equities for guaranteed government yields.
- **What it shows:** Compare each stock's annualized return during the 2023 tightening window against a 15.5% risk-free T-Bill benchmark.
- **Why it matters:** Even a stock that gained +5% during this window actually lost -10.5% in opportunity cost relative to risk-free government paper. This identifies the rare "super-resilient" stocks that beat fixed-income yields.
2. **Blue-Chip Foreign Exit Stress Test (Market Concentration Risk)** Historically, foreign investors account for over half of NSE trading activity and concentrate heavily in top liquid blue-chips (e.g., Safaricom, Equity Group, KCB, EABL, BAT). During 2023, Shilling depreciation toward KES 154/USD triggered heavy foreign capital flight.   
- **What it shows:** Segment your dataset into Top 5 Liquid Blue-Chips (by pre-crisis turnover) versus The Rest of the Market, and compare their drawdowns and turnover share.
- **Why it matters:** It proves whether the 2023 crash was a broad-based market collapse or a targeted foreign-exit liquidation concentrated in the most liquid counters.
3. **Recovery Velocity & "Rebound Elasticities" (2024–2025 Follow-Through)** Since our master dataset spans through 2025, we have the unique advantage of seeing how stocks recovered once the Shilling stabilized and the NSE 20 rebounded in 2024.
- **What it shows:** Measure the percentage gain from each stock's 2023 trough price (lowest price between May–Nov 2023) to its 2024/2025 peak or ending price, alongside Days-to-Recover (how many trading days it took to reclaim its May 2023 pre-crisis price).
- **Why it matters:** Distinguishes between temporary macro victims (quality stocks that snapped back sharply in 2024) and value traps (stocks that crashed in 2023 and never recovered).

In [10]:
# Ensure chronological sorting
df_master = df_master.sort_values(by=['Name', 'Date'])

# --- 1. T-BILL CROWDING-OUT & OPPORTUNITY COST (15.5% Benchmark) ---
# Calculate annualized return during the 7-month macro window (May - Nov 2023)
tbill_benchmark_annual = 15.5  # ~15.5% CBK T-Bill yield in late 2023
window_years = 214 / 365.25  # May 1 to Nov 30 is ~214 days

macro_detailed['Annualized_Window_Return_Pct'] = (
    (1 + (macro_detailed['Macro_Return_Pct'] / 100)) ** (1 / window_years) - 1
) * 100
macro_detailed['Excess_Return_vs_TBill_Pct'] = (
    macro_detailed['Annualized_Window_Return_Pct'] - tbill_benchmark_annual
)

tbill_beaters = macro_detailed[
    macro_detailed['Excess_Return_vs_TBill_Pct'] > 0
].sort_values(by='Excess_Return_vs_TBill_Pct', ascending=False)

print(
    '================================================================================'
)
print(
    f'  1. T-BILL CROWDING-OUT TEST: STOCKS BEATING 15.5% RISK-FREE RATE ({len(tbill_beaters)} STOCKS)  '
)
print(
    '================================================================================'
)
print(
    tbill_beaters[
        [
            'Name',
            'Sector',
            'Macro_Return_Pct',
            'Annualized_Window_Return_Pct',
            'Excess_Return_vs_TBill_Pct',
        ]
    ].to_string(index=False)
)


# --- 2. BLUE-CHIP FOREIGN EXIT STRESS TEST (Top 5 vs. Rest of Market) ---
# Identify Top 5 counters by pre-crisis baseline turnover (Jan-Apr 2023)
top5_names = (
    macro_detailed.sort_values(by='Baseline_Turnover_KES', ascending=False)
    .head(5)['Name']
    .tolist()
)
macro_detailed['Market_Tier'] = np.where(
    macro_detailed['Name'].isin(top5_names),
    'Top 5 Liquid Blue-Chips',
    'Rest of Market',
)

tier_comparison = (
    macro_detailed.groupby('Market_Tier')
    .agg(
        Company_Count=('Name', 'count'),
        Avg_Return_Pct=('Macro_Return_Pct', 'mean'),
        Median_Return_Pct=('Macro_Return_Pct', 'median'),
        Avg_Turnover_Change_Pct=('Turnover_Change_Pct', 'mean'),
        Total_Crisis_Daily_Turnover_KES=('Crisis_Turnover_KES', 'sum'),
    )
    .reset_index()
)

# Calculate share of total market daily turnover during the crisis
total_mkt_turnover = tier_comparison['Total_Crisis_Daily_Turnover_KES'].sum()
tier_comparison['Share_of_Total_Turnover_Pct'] = (
    tier_comparison['Total_Crisis_Daily_Turnover_KES'] / total_mkt_turnover
) * 100

print(
    '\n================================================================================'
)
print(
    '  2. FOREIGN EXIT STRESS TEST: TOP 5 BLUE-CHIPS VS. REST OF MARKET              '
)
print(
    '================================================================================'
)
print(f'Top 5 Liquid Counters: {top5_names}\n')
print(
    tier_comparison[
        [
            'Market_Tier',
            'Company_Count',
            'Avg_Return_Pct',
            'Median_Return_Pct',
            'Avg_Turnover_Change_Pct',
            'Share_of_Total_Turnover_Pct',
        ]
    ].to_string(index=False)
)


# --- 3. RECOVERY VELOCITY & REBOUND ELASTICITY (2023 Trough -> 2024/2025) ---
df_crisis_period = df_master[
    (df_master['Date'] >= '2023-05-01') & (df_master['Date'] <= '2023-11-30')
]
df_post_2023 = df_master[df_master['Date'] > '2023-11-30']

# Find lowest price (trough) and date during 2023 crisis per stock
idx_trough = df_crisis_period.groupby('Name')['Day Price'].idxmin()
troughs = df_crisis_period.loc[
    idx_trough, ['Name', 'Sector', 'Date', 'Day Price']
].rename(columns={'Date': 'Trough_Date', 'Day Price': 'Trough_Price_KES'})

# Find highest price achieved in 2024-2025 rebound window
post_peaks = (
    df_post_2023.groupby('Name')['Day Price']
    .max()
    .reset_index(name='Post_2023_Peak_KES')
)

recovery_df = troughs.merge(post_peaks, on='Name').merge(
    macro_detailed[['Name', 'Macro_Start_KES']], on='Name'
)

# Rebound % from 2023 bottom to 2024-2025 peak
recovery_df['Rebound_From_Trough_Pct'] = (
    (recovery_df['Post_2023_Peak_KES'] - recovery_df['Trough_Price_KES'])
    / recovery_df['Trough_Price_KES']
) * 100

# Check if stock fully recovered above its May 2023 pre-crisis price
recovery_df['Full_Recovery_Status'] = np.where(
    recovery_df['Post_2023_Peak_KES'] >= recovery_df['Macro_Start_KES'],
    'Fully Recovered',
    'Still Below May 2023',
)

print(
    '\n================================================================================'
)
print(
    '  3A. TOP 10 STRONGEST REBOUNDS FROM 2023 TROUGH (2024 - 2025 FOLLOW-THROUGH)   '
)
print(
    '================================================================================'
)
print(
    recovery_df.sort_values(by='Rebound_From_Trough_Pct', ascending=False)[
        [
            'Name',
            'Sector',
            'Macro_Start_KES',
            'Trough_Price_KES',
            'Post_2023_Peak_KES',
            'Rebound_From_Trough_Pct',
            'Full_Recovery_Status',
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print(
    '\n================================================================================'
)
print(
    '  3B. WEAKEST RECOVERIES / VALUE TRAPS (LOWEST REBOUND FROM 2023 TROUGH)        '
)
print(
    '================================================================================'
)
print(
    recovery_df.sort_values(by='Rebound_From_Trough_Pct', ascending=True)[
        [
            'Name',
            'Sector',
            'Macro_Start_KES',
            'Trough_Price_KES',
            'Post_2023_Peak_KES',
            'Rebound_From_Trough_Pct',
            'Full_Recovery_Status',
        ]
    ]
    .head(10)
    .to_string(index=False)
)

  1. T-BILL CROWDING-OUT TEST: STOCKS BEATING 15.5% RISK-FREE RATE (14 STOCKS)  
                             Name                   Sector  Macro_Return_Pct  Annualized_Window_Return_Pct  Excess_Return_vs_TBill_Pct
          Kapchorua Tea Kenya Plc             Agricultural         90.258449                    199.762616                  184.262616
         Eveready East Africa Ltd  Commercial and Services         67.948718                    142.284766                  126.784766
               Bamburi Cement Ltd  Construction and Allied         45.093946                     88.755268                   73.255268
               Kenya Orchards Ltd                      NaN         30.000000                     56.486092                   40.986092
         Williamson Tea Kenya Plc             Agricultural         25.270270                     46.894188                   31.394188
          CIC Insurance Group Ltd                Insurance         23.529412                     43.427175   

In [11]:

def compute_all_four_metrics(
    df, start_date, end_date, base_start=None, base_end=None, label='Shock'
):
  work = df.copy()
  work['Date'] = pd.to_datetime(work['Date'])
  work = work.sort_values(['Name', 'Date'])

  work['Daily_Return'] = work.groupby('Name')['Day Price'].pct_change()
  work['Daily_Turnover'] = work['Day Price'] * work['Volume']

  sub = work[(work['Date'] >= start_date) & (work['Date'] <= end_date)].copy()
  sectors = df[['Name', 'Sector']].drop_duplicates().set_index('Name')

  # 1. Window Return, Equal-Weighted & Turnover-Weighted Excess Return, and Cumulative Abnormal Return (CAR)
  first_p = sub.groupby('Name')['Day Price'].first()
  last_p = sub.groupby('Name')['Day Price'].last()
  win_ret_pct = ((last_p - first_p) / first_p) * 100

  eq_mkt_ret = win_ret_pct.mean()

  turn_by_stock = sub.groupby('Name')['Daily_Turnover'].sum()
  valid_turn_idx = turn_by_stock[turn_by_stock > 0].index
  tw_mkt_ret = (
      win_ret_pct.loc[valid_turn_idx] * turn_by_stock.loc[valid_turn_idx]
  ).sum() / turn_by_stock.loc[valid_turn_idx].sum()

  # Daily equal-weighted market return -> Cumulative Abnormal Return (CAR) over window
  daily_eq_ret = sub.groupby('Date')['Daily_Return'].mean()
  sub_with_mkt = sub.merge(
      daily_eq_ret.rename('Mkt_Daily_Ret'), on='Date', how='left'
  )
  sub_with_mkt['Abn_Ret'] = sub_with_mkt['Daily_Return'] - sub_with_mkt['Mkt_Daily_Ret']
  car_pct = sub_with_mkt.groupby('Name')['Abn_Ret'].sum() * 100

  # 2. Ulcer Index (RMS of % drawdowns from running peak) & Drawdown Share of Days
  cum_max = sub.groupby('Name')['Day Price'].cummax()
  dd_pct = ((sub['Day Price'] - cum_max) / cum_max) * 100
  ulcer_index = np.sqrt((dd_pct**2).groupby(sub['Name']).mean())
  pct_days_in_dd = (dd_pct < 0).groupby(sub['Name']).mean() * 100

  # 4. Amihud Illiquidity Ratio (scaled by 1e9 for readability)
  pos_turn = sub['Daily_Turnover'].where(sub['Daily_Turnover'] > 0)
  amihud_scaled = (sub['Daily_Return'].abs() / pos_turn) * 1e9
  amihud_mean = amihud_scaled.groupby(sub['Name']).mean()

  stock_df = pd.DataFrame({
      'Sector': sectors['Sector'],
      f'{label}_Win_Ret_Pct': win_ret_pct,
      f'{label}_Excess_EqWtd_Pct': win_ret_pct - eq_mkt_ret,
      f'{label}_Excess_TurnWtd_Pct': win_ret_pct - tw_mkt_ret,
      f'{label}_CAR_Pct': car_pct,
      f'{label}_Ulcer_Index': ulcer_index,
      f'{label}_Pct_Days_Underwater': pct_days_in_dd,
      f'{label}_Amihud_Illiq': amihud_mean,
      f'{label}_Total_Turnover': turn_by_stock,
  }).dropna(subset=[f'{label}_Win_Ret_Pct'])

  # 3. Liquidity Concentration Ratio (Top-5 and Top-10 share of total window turnover)
  def top_k_share(df_period, k):
    t = df_period.groupby('Name')['Daily_Turnover'].sum().sort_values(ascending=False)
    total = t.sum()
    return (t.head(k).sum() / total) * 100 if total > 0 else np.nan

  conc_summary = {
      'Window': f'{label} ({start_date} to {end_date})',
      'Shock_Top5_Turnover_Share_Pct': top_k_share(sub, 5),
      'Shock_Top10_Turnover_Share_Pct': top_k_share(sub, 10),
  }

  if base_start and base_end:
    base_sub = work[
        (work['Date'] >= base_start) & (work['Date'] <= base_end)
    ].copy()
    conc_summary['Base_Top5_Turnover_Share_Pct'] = top_k_share(base_sub, 5)
    conc_summary['Base_Top10_Turnover_Share_Pct'] = top_k_share(base_sub, 10)
    conc_summary['Top5_Share_Shift_pp'] = (
        conc_summary['Shock_Top5_Turnover_Share_Pct']
        - conc_summary['Base_Top5_Turnover_Share_Pct']
    )

  return stock_df.reset_index(), pd.DataFrame([conc_summary])


# Example usage on the 2023 Macro Tightening window vs. Jan-Apr 2023 baseline
stocks_2023, conc_2023 = compute_all_four_metrics(
    df_master,
    start_date='2023-05-01',
    end_date='2023-11-30',
    base_start='2023-01-01',
    base_end='2023-04-30',
    label='Macro2023',
)

print('--- Liquidity Concentration Shift ---')
print(conc_2023.to_string(index=False))

print('\n--- Top 10 Stocks by Cumulative Abnormal Return (CAR) ---')
print(
    stocks_2023.sort_values('Macro2023_CAR_Pct', ascending=False)
    .head(10)
    .to_string(index=False)
)

--- Liquidity Concentration Shift ---
                              Window  Shock_Top5_Turnover_Share_Pct  Shock_Top10_Turnover_Share_Pct  Base_Top5_Turnover_Share_Pct  Base_Top10_Turnover_Share_Pct  Top5_Share_Shift_pp
Macro2023 (2023-05-01 to 2023-11-30)                      78.346078                       89.896964                     85.557651                      95.297023            -7.211573

--- Top 10 Stocks by Cumulative Abnormal Return (CAR) ---
                            Name                  Sector  Macro2023_Win_Ret_Pct  Macro2023_Excess_EqWtd_Pct  Macro2023_Excess_TurnWtd_Pct  Macro2023_CAR_Pct  Macro2023_Ulcer_Index  Macro2023_Pct_Days_Underwater  Macro2023_Amihud_Illiq  Macro2023_Total_Turnover
         Kapchorua Tea Kenya Plc            Agricultural              90.258449                   91.603091                    105.119250          68.426773               9.601645                      66.206897              428.460076                21206150.0
        Eveready 

**Macro2023_Win_Ret_Pct:** Raw percentage price change (%) from the first trading day to the last trading day of the window.

**Macro2023_Excess_EqWtd_Pct:** Stock's window return minus the equal-weighted average window return across all stocks (%).

**Macro2023_Excess_TurnWtd_Pct:** Stock's window return minus the turnover-weighted average window return across the market (%).

**Macro2023_CAR_Pct:** Cumulative Abnormal Return (%)—the sum of daily abnormal returns (stock daily return - market daily average return) across the window.

**Macro2023_Ulcer_Index:** Root-mean-square of daily percentage drawdowns from the running peak inside the window; higher values mean deeper and/or longer drawdowns.

**Macro2023_Pct_Days_Underwater:** Percentage of trading days in the window where the stock's price sat below its prior running peak within that window.

**Macro2023_Amihud_Illiq:** Amihud Illiquidity Ratio (scaled by 
10
9
10 
9
 )—average ratio of |daily return| to daily KES turnover, showing how strongly price moves per shilling traded (higher = less liquid / more price impact).

**Macro2023_Total_Turnover:** Total KES value traded (Day Price * Volume) for that stock across the entire window.

## 4. The 2024 Finance Bill & Gen Z Protests (2024-06-01 to 2024-08-31).

**Macroeconomic & Market Backdrop (June – August 2024)**
**Unlike the prolonged multi-month grinding bear market of 2023, the mid-2024 shock was a sharp, sentiment-driven political and fiscal inflection point occurring right as Kenyan equities were beginning to recover from their 2023 lows:**

**Pre-Rejection Fiscal Shock (June 1 – June 25, 2024):** Widespread nationwide protests against proposed tax hikes in the Finance Bill 2024 triggered short-term political uncertainty, operational disruptions in major urban centers, and cautious institutional positioning.

**Post-Withdrawal Stabilization (June 26 – August 31, 2024):** Following the presidential withdrawal of the Finance Bill on June 26, immediate tax-hike fears subsided, though markets continued digesting fiscal deficit adjustments, sovereign credit rating reviews, and heightened domestic risk premiums.

In [12]:
# Ensure chronological sorting and required base columns
work_2024 = df_master.copy()
work_2024['Date'] = pd.to_datetime(work_2024['Date'])
work_2024 = work_2024.sort_values(['Name', 'Date'])

if 'Daily_Return' not in work_2024.columns:
  work_2024['Daily_Return'] = work_2024.groupby('Name')['Day Price'].pct_change()
if 'Daily_Turnover' not in work_2024.columns:
  work_2024['Daily_Turnover'] = work_2024['Day Price'] * work_2024['Volume']

# 1. Define Windows & Inflection Milestone
base_start, base_end = '2024-01-01', '2024-05-31'  # 2024 Rebound Baseline
shock_start, shock_end = '2024-06-01', '2024-08-31'  # Gen Z Protest & Fiscal Window
inflection_date = '2024-06-26'  # Finance Bill 2024 Withdrawal

df_base_24 = work_2024[
    (work_2024['Date'] >= base_start) & (work_2024['Date'] <= base_end)
].copy()
df_shock_24 = work_2024[
    (work_2024['Date'] >= shock_start) & (work_2024['Date'] <= shock_end)
].copy()

df_peak_protest = df_shock_24[df_shock_24['Date'] < inflection_date].copy()
df_post_bill = df_shock_24[df_shock_24['Date'] >= inflection_date].copy()


# Helper function for sub-window price changes
def get_subwindow_returns(df_sub, prefix):
  df_sub = df_sub.sort_values(['Name', 'Date'])
  first_p = (
      df_sub.groupby('Name')['Day Price']
      .first()
      .reset_index(name=f'{prefix}_Start_KES')
  )
  last_p = (
      df_sub.groupby('Name')['Day Price']
      .last()
      .reset_index(name=f'{prefix}_End_KES')
  )
  res = pd.merge(first_p, last_p, on='Name')
  res[f'{prefix}_Change_KES'] = (
      res[f'{prefix}_End_KES'] - res[f'{prefix}_Start_KES']
  )
  res[f'{prefix}_Return_Pct'] = (
      res[f'{prefix}_Change_KES'] / res[f'{prefix}_Start_KES']
  ) * 100
  return res


# 2. Compute Overall, Peak Protest (Jun 1–25), and Post-Withdrawal (Jun 26–Aug 31) Returns
overall_24 = get_subwindow_returns(df_shock_24, 'GenZ24')
peak_24 = get_subwindow_returns(df_peak_protest, 'PeakProtest')
post_24 = get_subwindow_returns(df_post_bill, 'PostBill')

# 3. Compute Advanced Stress Metrics (CAR, Ulcer Index, Amihud Illiquidity, Turnover Shift)
daily_eq_mkt = df_shock_24.groupby('Date')['Daily_Return'].mean()
df_shock_24 = df_shock_24.merge(
    daily_eq_mkt.rename('Mkt_Daily_Ret'), on='Date', how='left'
)
df_shock_24['Abn_Ret'] = (
    df_shock_24['Daily_Return'] - df_shock_2024['Mkt_Daily_Ret']
    if 'df_shock_2024' in locals()
    else df_shock_24['Daily_Return'] - df_shock_24['Mkt_Daily_Ret']
)
car_24 = (
    df_shock_24.groupby('Name')['Abn_Ret'].sum() * 100
).reset_index(name='GenZ24_CAR_Pct')

cum_max_24 = df_shock_24.groupby('Name')['Day Price'].cummax()
dd_pct_24 = ((df_shock_24['Day Price'] - cum_max_24) / cum_max_24) * 100
ulcer_24 = (
    np.sqrt((dd_pct_24**2).groupby(df_shock_24['Name']).mean())
    .reset_index(name='GenZ24_Ulcer_Index')
)

pos_turn_24 = df_shock_24['Daily_Turnover'].where(
    df_shock_24['Daily_Turnover'] > 0
)
amihud_24 = (
    ((df_shock_24['Daily_Return'].abs() / pos_turn_24) * 1e9)
    .groupby(df_shock_24['Name'])
    .mean()
    .reset_index(name='GenZ24_Amihud_Illiq')
)

base_turn_24 = (
    df_base_24.groupby('Name')['Daily_Turnover']
    .mean()
    .reset_index(name='Base_Daily_Turn_KES')
)
shock_turn_24 = (
    df_shock_24.groupby('Name')['Daily_Turnover']
    .mean()
    .reset_index(name='Shock_Daily_Turn_KES')
)

# 4. Merge Master 2024 Protest Table
sectors = df_master[['Name', 'Sector']].drop_duplicates()
genz_detailed = (
    sectors.merge(overall_24, on='Name')
    .merge(peak_24[['Name', 'PeakProtest_Return_Pct']], on='Name', how='left')
    .merge(post_24[['Name', 'PostBill_Return_Pct']], on='Name', how='left')
    .merge(car_24, on='Name', how='left')
    .merge(ulcer_24, on='Name', how='left')
    .merge(amihud_24, on='Name', how='left')
    .merge(base_turn_24, on='Name', how='left')
    .merge(shock_turn_24, on='Name', how='left')
)

genz_detailed['Turnover_Change_Pct'] = (
    (genz_detailed['Shock_Daily_Turn_KES'] - genz_detailed['Base_Daily_Turn_KES'])
    / genz_detailed['Base_Daily_Turn_KES']
) * 100

# 5. Liquidity Concentration Check (Flight to Blue-Chips)
def top_k_share(df_p, k):
  t = df_p.groupby('Name')['Daily_Turnover'].sum().sort_values(ascending=False)
  return (t.head(k).sum() / t.sum()) * 100 if t.sum() > 0 else np.nan


conc_24 = pd.DataFrame([{
    'Window': '2024 Gen Z Protests (Jun–Aug)',
    'Base_Top5_Share_Pct': top_k_share(df_base_24, 5),
    'Shock_Top5_Share_Pct': top_k_share(df_shock_24, 5),
    'Top5_Share_Shift_pp': top_k_share(df_shock_24, 5)
    - top_k_share(df_base_24, 5),
    'Base_Top10_Share_Pct': top_k_share(df_base_24, 10),
    'Shock_Top10_Share_Pct': top_k_share(df_shock_24, 10),
}])

display_cols = [
    'Name',
    'Sector',
    'GenZ24_Start_KES',
    'GenZ24_End_KES',
    'GenZ24_Return_Pct',
    'PeakProtest_Return_Pct',
    'PostBill_Return_Pct',
    'GenZ24_CAR_Pct',
    'GenZ24_Ulcer_Index',
    'Turnover_Change_Pct',
]

print('==============================================================================================================')
print('             2024 GEN Z PROTESTS: LIQUIDITY CONCENTRATION SHIFT VS. JAN–MAY 2024 BASELINE                     ')
print('==============================================================================================================')
print(conc_24.to_string(index=False))

print('\n==============================================================================================================')
print('             2024 GEN Z PROTESTS: TOP 10 BEST PERFORMING COMPANIES (MOST RESILIENT)                           ')
print('==============================================================================================================')
print(
    genz_detailed.sort_values('GenZ24_Return_Pct', ascending=False)[
        display_cols
    ]
    .head(10)
    .to_string(index=False)
)

print('\n==============================================================================================================')
print('             2024 GEN Z PROTESTS: TOP 10 WORST PERFORMING COMPANIES (DEEPEST DROPS)                           ')
print('==============================================================================================================')
print(
    genz_detailed.sort_values('GenZ24_Return_Pct', ascending=True)[display_cols]
    .head(10)
    .to_string(index=False)
)

# 6. Sector-Level Summary
sector_genz = (
    genz_detailed.groupby('Sector')
    .agg(
        Companies=('Name', 'count'),
        Avg_Overall_Ret_Pct=('GenZ24_Return_Pct', 'mean'),
        Avg_PeakProtest_Ret_Pct=('PeakProtest_Return_Pct', 'mean'),
        Avg_PostBill_Ret_Pct=('PostBill_Return_Pct', 'mean'),
        Avg_CAR_Pct=('GenZ24_CAR_Pct', 'mean'),
        Avg_Ulcer_Index=('GenZ24_Ulcer_Index', 'mean'),
        Avg_Turnover_Change_Pct=('Turnover_Change_Pct', 'mean'),
    )
    .sort_values('Avg_Overall_Ret_Pct', ascending=False)
    .reset_index()
)

print('\n==============================================================================================================')
print('             2024 GEN Z PROTESTS: SECTOR-LEVEL RESILIENCE & INFLECTION SUMMARY                                ')
print('==============================================================================================================')
print(sector_genz.to_string(index=False))

             2024 GEN Z PROTESTS: LIQUIDITY CONCENTRATION SHIFT VS. JAN–MAY 2024 BASELINE                     
                       Window  Base_Top5_Share_Pct  Shock_Top5_Share_Pct  Top5_Share_Shift_pp  Base_Top10_Share_Pct  Shock_Top10_Share_Pct
2024 Gen Z Protests (Jun–Aug)            85.285661              74.04995           -11.235712             93.143295              91.227883

             2024 GEN Z PROTESTS: TOP 10 BEST PERFORMING COMPANIES (MOST RESILIENT)                           
                                    Name                  Sector  GenZ24_Start_KES  GenZ24_End_KES  GenZ24_Return_Pct  PeakProtest_Return_Pct  PostBill_Return_Pct  GenZ24_CAR_Pct  GenZ24_Ulcer_Index  Turnover_Change_Pct
                      Bamburi Cement Ltd Construction and Allied             43.30           77.50          78.983834               -2.193995            81.711606       70.813810            1.900075           170.818891
                       Express Kenya Plc Commercial and Ser

**PeakProtest_Return_Pct (Jun 1 – Jun 25):** Captures immediate market reaction during peak street protests and urban business disruptions prior to the Finance Bill withdrawal.

**PostBill_Return_Pct (Jun 26 – Aug 31):** Measures how sectors behaved after the bill was dropped—revealing whether relief from halted tax hikes outweighed concerns over sovereign credit downgrades and fiscal revenue gaps.

**GenZ24_Ulcer_Index:** Shows whether price drops were brief, sharp panics (low Ulcer Index) or sustained multi-week declines (high Ulcer Index).

**Top5_Share_Shift_pp:** Confirms whether institutional investors crowded into the top 5 liquid blue-chips during political unrest.

In [13]:

def build_election_playbook(df, shock_windows):
  """
    shock_windows: list of tuples -> [(label, start_date, end_date), ...]
    Builds a composite resilience vs. fragility score per stock from historical shocks.
    """
  work = df.copy()
  work['Date'] = pd.to_datetime(work['Date'])
  work = work.sort_values(['Name', 'Date'])
  work['Daily_Return'] = work.groupby('Name')['Day Price'].pct_change()
  work['Daily_Turnover'] = work['Day Price'] * work['Volume']

  # Overall liquidity & activity baseline across full dataset
  full_stats = (
      work.groupby('Name')
      .agg(
          Sector=('Sector', 'first'),
          Avg_Daily_Turnover=('Daily_Turnover', 'mean'),
          Zero_Vol_Share_Pct=(
              'Volume',
              lambda s: (s.fillna(0) == 0).mean() * 100,
          ),
          Latest_Price_KES=('Day Price', 'last'),
      )
      .reset_index()
  )

  per_window_tables = []

  for label, s_date, e_date in shock_windows:
    sub = work[(work['Date'] >= s_date) & (work['Date'] <= e_date)].copy()
    if sub.empty:
      continue

    first_p = sub.groupby('Name')['Day Price'].first()
    last_p = sub.groupby('Name')['Day Price'].last()
    win_ret = ((last_p - first_p) / first_p) * 100

    # Cumulative Abnormal Return (CAR) vs equal-weighted daily market return
    mkt_daily = sub.groupby('Date')['Daily_Return'].mean()
    sub = sub.merge(mkt_daily.rename('Mkt_Ret'), on='Date', how='left')
    sub['Abn_Ret'] = sub['Daily_Return'] - sub['Mkt_Ret']
    car = sub.groupby('Name')['Abn_Ret'].sum() * 100

    # Ulcer Index (RMS of % drawdowns from running peak)
    cum_max = sub.groupby('Name')['Day Price'].cummax()
    dd_pct = ((sub['Day Price'] - cum_max) / cum_max) * 100
    ulcer = np.sqrt((dd_pct**2).groupby(sub['Name']).mean())

    # Amihud Illiquidity (scaled by 1e9)
    pos_turn = sub['Daily_Turnover'].where(sub['Daily_Turnover'] > 0)
    amihud = ((sub['Daily_Return'].abs() / pos_turn) * 1e9).groupby(
        sub['Name']
    ).mean()

    w_df = pd.DataFrame({
        'Name': win_ret.index,
        f'{label}_Ret_Pct': win_ret.values,
        f'{label}_CAR_Pct': car.reindex(win_ret.index).values,
        f'{label}_Ulcer': ulcer.reindex(win_ret.index).values,
        f'{label}_Amihud': amihud.reindex(win_ret.index).values,
    })
    per_window_tables.append(w_df)

  scored = full_stats.copy()
  for tbl in per_window_tables:
    scored = scored.merge(tbl, on='Name', how='left')

  ret_cols = [c for c in scored.columns if c.endswith('_Ret_Pct')]
  car_cols = [c for c in scored.columns if c.endswith('_CAR_Pct')]
  ulcer_cols = [c for c in scored.columns if c.endswith('_Ulcer')]
  amihud_cols = [c for c in scored.columns if c.endswith('_Amihud')]

  # Cross-shock averages
  scored['Avg_Shock_Ret_Pct'] = scored[ret_cols].mean(axis=1)
  scored['Avg_Shock_CAR_Pct'] = scored[car_cols].mean(axis=1)
  scored['Avg_Shock_Ulcer'] = scored[ulcer_cols].mean(axis=1)
  scored['Avg_Shock_Amihud'] = scored[amihud_cols].mean(axis=1)

  # Z-score helper (handles constant/all-NaN columns safely)
  def z_score(series, invert=False):
    s = series.fillna(series.median())
    std = s.std()
    if pd.isna(std) or std == 0:
      z = pd.Series(0.0, index=series.index)
    else:
      z = (s - s.mean()) / std
    return -z if invert else z

  # Higher Resilience Score = stronger shock returns/CAR, lower Ulcer pain, lower illiquidity & zero-volume friction
  scored['Resilience_Score'] = (
      0.35 * z_score(scored['Avg_Shock_CAR_Pct'])
      + 0.25 * z_score(scored['Avg_Shock_Ret_Pct'])
      + 0.20 * z_score(scored['Avg_Shock_Ulcer'], invert=True)
      + 0.10 * z_score(scored['Avg_Shock_Amihud'], invert=True)
      + 0.10 * z_score(scored['Zero_Vol_Share_Pct'], invert=True)
  ).round(3)

  # Higher Fragility Score = deeper shock losses/negative CAR, high Ulcer pain, high illiquidity/zero-volume drag
  scored['Fragility_Score'] = (
      0.35 * z_score(scored['Avg_Shock_Ulcer'])
      + 0.25 * z_score(scored['Avg_Shock_CAR_Pct'], invert=True)
      + 0.20 * z_score(scored['Avg_Shock_Ret_Pct'], invert=True)
      + 0.20 * z_score(scored['Zero_Vol_Share_Pct'])
  ).round(3)

  # Assign Playbook Bucket
  def assign_bucket(row):
    if row['Zero_Vol_Share_Pct'] > 40 or row['Fragility_Score'] > 1.0:
      return 'Avoid / High Fragility Risk'
    elif row['Resilience_Score'] >= 0.75:
      return 'Favor / Shock-Resilient Candidate'
    elif row['Avg_Daily_Turnover'] >= scored['Avg_Daily_Turnover'].quantile(
        0.75
    ):
      return 'Liquid Core / Watch Volatility'
    else:
      return 'Selective / Neutral'

  scored['Playbook_Bucket'] = scored.apply(assign_bucket, axis=1)
  return scored.sort_values('Resilience_Score', ascending=False)


windows_to_score = [
    ('Covid20', '2020-03-01', '2020-07-31'),
    ('Elec22', '2022-06-01', '2022-10-31'),
    ('Macro23', '2023-05-01', '2023-11-30'),
    ('GenZ24', '2024-06-01', '2024-08-31'),
]

playbook_df = build_election_playbook(df_master, windows_to_score)

cols_out = [
    'Name',
    'Sector',
    'Playbook_Bucket',
    'Resilience_Score',
    'Fragility_Score',
    'Avg_Shock_Ret_Pct',
    'Avg_Shock_CAR_Pct',
    'Avg_Shock_Ulcer',
    'Zero_Vol_Share_Pct',
]

print('=== TOP 10 RESILIENT CANDIDATES (HIGHER RESILIENCE SCORE) ===')
print(playbook_df[cols_out].head(10).to_string(index=False))

print('\n=== TOP 10 HIGHER-FRAGILITY / AVOID FLAGS (HIGHEST FRAGILITY SCORE) ===')
print(
    playbook_df.sort_values('Fragility_Score', ascending=False)[cols_out]
    .head(10)
    .to_string(index=False)
)

print('\n=== PLAYBOOK BUCKET COUNTS ===')
print(playbook_df['Playbook_Bucket'].value_counts().to_string())

# Export for Tableau / Capstone report
playbook_df.to_csv('nse_election_2027_playbook.csv', index=False)

=== TOP 10 RESILIENT CANDIDATES (HIGHER RESILIENCE SCORE) ===
                                Name                   Sector                   Playbook_Bucket  Resilience_Score  Fragility_Score  Avg_Shock_Ret_Pct  Avg_Shock_CAR_Pct  Avg_Shock_Ulcer  Zero_Vol_Share_Pct
             Kapchorua Tea Kenya Plc             Agricultural       Avoid / High Fragility Risk             1.878           -1.409          30.420393          27.514125         9.140112           46.702002
                   Kenya Airways Plc  Commercial and Services       Avoid / High Fragility Risk             1.363           -1.006          19.534884          20.129344         7.735039           65.783274
            Williamson Tea Kenya Plc             Agricultural Favor / Shock-Resilient Candidate             1.276           -0.992          14.264561          20.486597        10.156309           17.844523
                   Sameer Africa Plc  Commercial and Services Favor / Shock-Resilient Candidate             1.159 